In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Times New Roman"

from utils import *

In [ ]:
TARGETS_LIST_PATH = "targets/targets_list.csv"
targets_df = pd.read_csv(TARGETS_LIST_PATH)

pdb_to_af = dict(zip(targets_df["PDB_ID"], targets_df["AF_ID"]))
af_to_pdb = dict(zip(targets_df["AF_ID"], targets_df["PDB_ID"]))

In [ ]:
OUTPUT_DIR = "analysis/pocket_comparison/outputs"
GLOBAL_STATS_PATH = os.path.join(OUTPUT_DIR, "global/global_statistics.csv")
GLOBAL_PAIRS_PATH = os.path.join(OUTPUT_DIR, "global/global_pocket_pairs.csv")
GLOBAL_PROTEIN_PATH = os.path.join(OUTPUT_DIR, "global/global_protein_summary.csv")

#### General statistics.

In [ ]:
df = pd.read_csv(GLOBAL_PAIRS_PATH)
global_stats = pd.read_csv(GLOBAL_STATS_PATH)
stats_dict = global_stats.set_index("metric")["value"].to_dict()

print(f"Number of protein pairs analyzed: {int(stats_dict["n_protein_pairs"])}")

In [ ]:
n_pair = int(stats_dict["n_pair_rows"])
n_matched = int(stats_dict["n_matched"])
n_weak_matches = int(stats_dict["n_weak_matches"])
n_pdb_only = int(stats_dict["n_pdb_only"])
n_af_only = int(stats_dict["n_af_only"])

print(f"Total number of pocket comparison records: {n_pair}")
print(f"Number of matched pocket pairs: {n_matched}")
print(f"Number of weakly matched pocket pairs: {n_weak_matches}")
print(f"Number of pockets found only in PDB structures: {n_pdb_only}")
print(f"Number of pockets found only in AlphaFold structures: {n_af_only}")

#### Matching results.

In [ ]:
labels = ["Matched", "Weak match", "PDB only", "AF only"]
sizes = [n_matched, n_weak_matches, n_pdb_only, n_af_only]

fig, ax = plt.subplots(figsize=(4, 4))

ax.pie(
    sizes,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
)

ax.set_title("Pocket matching outcomes")
plt.show()

In [ ]:
pockets_df, pockets_pdb, pockets_af = prepare_pockets_df(df)

print(f"Total number of pockets: {pockets_df.shape[0]}")
print(f"Number of PDB pockets: {pockets_pdb.shape[0]}")
print(f"Number of AF pockets: {pockets_af.shape[0]}")

In [ ]:
viz_status(pockets_pdb, pockets_af)

#### `fpocket` *pocket score* and *druggability score* for detected pockets.

In [ ]:
show_score_stats(pockets_pdb, pockets_af)

In [ ]:
plot_distribution(pockets_pdb, pockets_af, metric="Score")
plot_distribution(pockets_pdb, pockets_af,  metric="Drug")

#### Quantile thresholds for pocket score and druggability score in PDB structures and AlphaFold models. 

For each quantile **q**, the corresponding `Score` and `Drug` values represent **the minimum thresholds required to belong to the top (1 − q)** fraction of pockets. 

The `Count` columns indicate the number of pockets with both `Score` and `Drug` values greater than or equal to their respective quantile thresholds.

In [ ]:
quantiles = [0.5, 0.75, 0.8, 0.9, 0.91, 0.92, 0.93, 0.94, 0.95]
show_score_quantiles(pockets_pdb, pockets_af, quantiles=quantiles)

#### Filtering and analysing the best rated pockets.

You can adjust the thresholds:
- `min_score`
- `min_drug`
- `jaccard_threshold`

In [ ]:
min_score = 0.15
min_drug = 0.1
jaccard_threshold = 0.3

In [ ]:
filtered_pdb = score_filter(pockets_pdb, min_score=min_score, min_drug=min_drug)
filtered_af = score_filter(pockets_af, min_score=min_score, min_drug=min_drug)

pdb_structures = filtered_pdb["ID"].nunique()
af_models = filtered_af["ID"].nunique()

print(f"Number of pockets after filtering:")
print(f"    PDB: {filtered_pdb.shape[0]}")
print(f"    AF:  {filtered_af.shape[0]}")

print(f"Remaining pockets came from:")
print(f"    {pdb_structures} PDB structures")
print(f"    {af_models} AF models")

In [ ]:
matches_results_pdb = analyse_matches_pdb(filtered_pdb, filtered_af, pdb_to_af, jaccard_threshold)

In [ ]:
matches_results_af = analyse_matches_af(filtered_pdb, filtered_af, af_to_pdb, jaccard_threshold)

In [ ]:
_, results_df = matches_results_af
retained_df = results_df[results_df["Partner retained"]]

retained_pairs = identify_retained_pairs(retained_df, filtered_pdb, af_to_pdb)
retained_pairs.round(2).sort_values(["Jaccard"], ascending=False)

#### Origin of detected pockets.

Using group label (`GROUP`) from `targets_list.csv` which classifies the poteins as:
- `1` – likely to contain druggable small-molecule binding pockets
- `0` – control group; likely to lack druggable pockets or contain only small and/or poorly accessible pockets

In [ ]:
drug_origin_pdb = analyze_origin(pockets_pdb, targets_df)
drug_origin_af = analyze_origin(pockets_af, targets_df, mode="AF")

print(f"Percent of pockets in \"druggable\" proteins:")
print(f"    PDB: {drug_origin_pdb:.2f}%")
print(f"    AF:  {drug_origin_af:.2f}%")

In [ ]:
drug_origin_pdb_filtered = analyze_origin(filtered_pdb, targets_df)
drug_origin_af_filtered = analyze_origin(filtered_af, targets_df, mode="AF")

print(f"Percent of filtered pockets in \"druggable\" proteins:")
print(f"    PDB: {drug_origin_pdb_filtered:.2f}%")
print(f"    AF:  {drug_origin_af_filtered:.2f}%")

In [ ]:
score_r = stats_corr(df, targets_df)

In [ ]:
drug_r = stats_corr(df, targets_df, stats="drug")

#### pLDDT impact

In [ ]:
plddt_summary, _ = plddt_impact(pockets_af)
plddt_summary